<a href="https://colab.research.google.com/github/gagandeep02/Data-Visualisation/blob/main/My_AI_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import time
import difflib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


# Keep your uploaded Kaggle CSV in the same Colab folder.
DATASET_PATH = "customer_support_tickets.csv"

# Free instruction-tuned LLM. Use "google/flan-t5-base" for better quality if Colab is slow enough.
LLM_MODEL_NAME = "google/flan-t5-small"

RANDOM_STATE = 42
TEST_SIZE = 0.25

# Set None for full test set. 300 is faster for Colab demo/evaluation.
EVALUATION_SAMPLE_SIZE = 300

CONFIDENCE_THRESHOLD = 0.60
SIMULATED_PARTICIPANTS = 30


def normalize_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )
    return df


def find_column(df, possible_names):
    for name in possible_names:
        if name in df.columns:
            return name
    raise ValueError(
        "Required column not found. Tried: "
        + ", ".join(possible_names)
        + "\nAvailable columns: "
        + ", ".join(df.columns)
    )


def load_and_prepare_data(path):
    if not Path(path).exists():
        csv_files = sorted(Path.cwd().glob("*.csv"))
        if len(csv_files) == 1:
            path = str(csv_files[0])
            print("Using detected CSV file:", path)
        else:
            raise FileNotFoundError(
                f"Dataset file not found: {path}\n"
                "Upload customer_support_tickets.csv or update DATASET_PATH.\n"
                "CSV files found: " + ", ".join(str(file) for file in csv_files)
            )

    df = pd.read_csv(path)
    df = normalize_columns(df)

    subject_col = find_column(df, [
        "ticket_subject",
        "subject",
        "title",
        "issue_subject",
    ])
    description_col = find_column(df, [
        "ticket_description",
        "description",
        "issue_description",
        "text",
        "query",
        "message",
    ])
    type_col = find_column(df, [
        "ticket_type",
        "type",
        "category",
        "intent",
        "issue_type",
    ])
    priority_col = find_column(df, [
        "ticket_priority",
        "priority",
        "urgency",
    ])

    optional_cols = []
    for col in ["ticket_status", "status", "customer_satisfaction_rating", "satisfaction_rating"]:
        if col in df.columns:
            optional_cols.append(col)

    df["input_text"] = (
        df[subject_col].fillna("").astype(str)
        + " "
        + df[description_col].fillna("").astype(str)
    )

    keep_cols = ["input_text", type_col, priority_col] + optional_cols
    data = df[keep_cols].copy()

    rename_map = {
        type_col: "ticket_type",
        priority_col: "ticket_priority",
    }
    if "status" in data.columns:
        rename_map["status"] = "ticket_status"
    if "satisfaction_rating" in data.columns:
        rename_map["satisfaction_rating"] = "customer_satisfaction_rating"

    data = data.rename(columns=rename_map)
    data["input_text"] = data["input_text"].fillna("").astype(str)
    data["ticket_type"] = data["ticket_type"].fillna("").astype(str)
    data["ticket_priority"] = data["ticket_priority"].fillna("").astype(str)

    data = data[data["input_text"].str.strip().str.len() > 5]
    data = data[data["ticket_type"].str.strip().str.len() > 0]

    class_counts = data["ticket_type"].value_counts()
    valid_classes = class_counts[class_counts >= 2].index
    data = data[data["ticket_type"].isin(valid_classes)].reset_index(drop=True)

    return data


def load_llm():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(LLM_MODEL_NAME).to(device)
    model.eval()

    print("LLM loaded:", LLM_MODEL_NAME)
    print("Device:", device)
    return {
        "tokenizer": tokenizer,
        "model": model,
        "device": device,
    }


def generate_text(llm, prompt, max_new_tokens):
    tokenizer = llm["tokenizer"]
    model = llm["model"]
    device = llm["device"]

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


def make_classification_prompt(ticket_text, priority, labels):
    label_text = ", ".join(labels)
    return f"""
You are an LLM-powered AI agent for business process automation.
Your task is to classify a customer support ticket into exactly one ticket type.

Allowed ticket types:
{label_text}

Ticket priority: {priority}
Ticket text: {ticket_text}

Return only one ticket type from the allowed list.
"""


def normalize_prediction(raw_prediction, labels):
    raw = str(raw_prediction).strip()
    raw_lower = raw.lower()

    for label in labels:
        if raw_lower == label.lower():
            return label

    for label in labels:
        if label.lower() in raw_lower or raw_lower in label.lower():
            return label

    close = difflib.get_close_matches(raw, labels, n=1, cutoff=0.0)
    return close[0] if close else labels[0]


def pseudo_confidence(raw_prediction, selected_label):
    raw = str(raw_prediction).strip().lower()
    label = str(selected_label).strip().lower()

    if raw == label:
        return 0.95
    if label in raw or raw in label:
        return 0.80

    similarity = difflib.SequenceMatcher(None, raw, label).ratio()
    return round(max(0.30, min(0.75, similarity)), 4)


def llm_classify_ticket(llm, ticket_text, priority, labels):
    prompt = make_classification_prompt(ticket_text, priority, labels)
    output = generate_text(llm, prompt, max_new_tokens=20)
    predicted_label = normalize_prediction(output, labels)
    confidence = pseudo_confidence(output, predicted_label)
    return predicted_label, confidence, output


def business_action(predicted_type, priority, confidence):
    priority_text = str(priority).lower()

    if confidence < CONFIDENCE_THRESHOLD:
        review_action = "Human review required before automation"
    else:
        review_action = "Automation allowed"

    if "critical" in priority_text or "high" in priority_text:
        urgency_action = "Escalate to senior support team"
    elif "medium" in priority_text:
        urgency_action = "Assign to standard support queue"
    else:
        urgency_action = "Assign to normal support queue"

    return f"{review_action}; {urgency_action}; route to {predicted_type} workflow"


def error_recovery_action(failure_type, predicted_type, confidence):
    if failure_type == "success":
        return "Proceed with automated workflow and log the decision for audit."

    if failure_type == "low_confidence_success":
        return (
            "Prediction matches the expected class but confidence is low. "
            "Ask one clarifying question, then route to the predicted workflow after confirmation."
        )

    if failure_type == "wrong_class_low_confidence":
        return (
            "Do not execute the automated action. Send ticket to human review, "
            "request missing details, and store the case as a low-confidence failure example."
        )

    if failure_type == "wrong_class_high_confidence":
        return (
            "Critical failure: block automation, escalate to supervisor, "
            "log high-confidence misclassification, and update evaluation records."
        )

    return "Send to human review and log unknown failure."


def make_response_prompt(ticket_text, predicted_type, priority, recovery_action):
    return f"""
You are an LLM-powered business support agent.
Write a short professional customer-support response.

Ticket type: {predicted_type}
Priority: {priority}
Ticket text: {ticket_text}
Internal recovery/action instruction: {recovery_action}

Response must be polite, concise, and business appropriate.
"""


def generate_customer_response(llm, ticket_text, predicted_type, priority, recovery_action):
    prompt = make_response_prompt(ticket_text, predicted_type, priority, recovery_action)
    output = generate_text(llm, prompt, max_new_tokens=80)
    return str(output).strip()


def failure_mode(true_type, predicted_type, confidence):
    if true_type == predicted_type and confidence >= CONFIDENCE_THRESHOLD:
        return "success"
    if true_type == predicted_type and confidence < CONFIDENCE_THRESHOLD:
        return "low_confidence_success"
    if true_type != predicted_type and confidence >= CONFIDENCE_THRESHOLD:
        return "wrong_class_high_confidence"
    return "wrong_class_low_confidence"


def reliability_score(mode):
    scores = {
        "success": 1.00,
        "low_confidence_success": 0.75,
        "wrong_class_low_confidence": 0.35,
        "wrong_class_high_confidence": 0.00,
    }
    return scores.get(mode, 0.00)


def trust_label(score):
    if score >= 0.80:
        return "high_trust"
    if score >= 0.50:
        return "medium_trust"
    return "low_trust"


def evaluate_llm_agent(data, llm):
    class_count = data["ticket_type"].nunique()
    test_size_count = int(len(data) * TEST_SIZE)
    can_stratify = class_count > 1 and test_size_count >= class_count
    stratify_target = data["ticket_type"] if can_stratify else None

    train_df, test_df = train_test_split(
        data,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=stratify_target,
    )

    if EVALUATION_SAMPLE_SIZE is not None and len(test_df) > EVALUATION_SAMPLE_SIZE:
        test_df = test_df.sample(EVALUATION_SAMPLE_SIZE, random_state=RANDOM_STATE)

    labels = sorted(train_df["ticket_type"].unique())
    results = []

    print("\nEvaluating LLM-powered AI agent...")
    print("Allowed ticket types:", labels)
    print("Evaluation records:", len(test_df))

    start_time = time.time()

    for index, row in test_df.reset_index(drop=True).iterrows():
        predicted_type, confidence, raw_llm_output = llm_classify_ticket(
            llm=llm,
            ticket_text=row["input_text"],
            priority=row["ticket_priority"],
            labels=labels,
        )

        mode = failure_mode(row["ticket_type"], predicted_type, confidence)
        score = reliability_score(mode)
        recovery = error_recovery_action(mode, predicted_type, confidence)

        results.append({
            "input_text": row["input_text"],
            "ticket_priority": row["ticket_priority"],
            "true_ticket_type": row["ticket_type"],
            "llm_raw_output": raw_llm_output,
            "predicted_ticket_type": predicted_type,
            "confidence": confidence,
            "business_action": business_action(predicted_type, row["ticket_priority"], confidence),
            "failure_mode": mode,
            "error_recovery_action": recovery,
            "reliability_score": score,
            "human_trust_estimate": trust_label(score),
        })

        if (index + 1) % 25 == 0:
            print(f"Processed {index + 1}/{len(test_df)} tickets")

    total_time = time.time() - start_time
    avg_latency = total_time / len(test_df)

    result_df = pd.DataFrame(results)

    accuracy = accuracy_score(result_df["true_ticket_type"], result_df["predicted_ticket_type"])
    task_success_rate = (result_df["failure_mode"] == "success").mean()
    failure_rate = 1 - task_success_rate
    average_reliability = result_df["reliability_score"].mean()

    print("\n========== LLM AGENT DATASET SUMMARY ==========")
    print("Dataset used: Customer Support Ticket Dataset")
    print("Total records used:", len(data))
    print("Training/reference records:", len(train_df))
    print("Evaluated records:", len(test_df))
    print("Target column: ticket_type")
    print("Business task: LLM ticket classification, workflow routing, reliability evaluation")

    print("\n========== LLM AGENT PERFORMANCE ==========")
    print("Accuracy:", round(accuracy, 4))
    print("Task success rate:", round(task_success_rate, 4))
    print("Failure rate:", round(failure_rate, 4))
    print("Average reliability score:", round(average_reliability, 4))
    print("Average latency per ticket:", round(avg_latency, 4), "seconds")

    print("\n========== FAILURE MODES ==========")
    print(result_df["failure_mode"].value_counts())

    print("\n========== HUMAN TRUST ESTIMATE ==========")
    print(result_df["human_trust_estimate"].value_counts())

    print("\n========== CLASSIFICATION REPORT ==========")
    print(classification_report(
        result_df["true_ticket_type"],
        result_df["predicted_ticket_type"],
        zero_division=0,
    ))

    all_labels = sorted(set(result_df["true_ticket_type"]) | set(result_df["predicted_ticket_type"]))
    cm = confusion_matrix(
        result_df["true_ticket_type"],
        result_df["predicted_ticket_type"],
        labels=all_labels,
    )
    cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)

    result_df.to_csv("llm_agent_reliability_evaluation_results.csv", index=False)
    cm_df.to_csv("llm_ticket_type_confusion_matrix.csv")

    summary = {
        "dataset": "Customer Support Ticket Dataset",
        "llm_model": LLM_MODEL_NAME,
        "records_used": int(len(data)),
        "evaluated_records": int(len(test_df)),
        "accuracy": float(round(accuracy, 4)),
        "task_success_rate": float(round(task_success_rate, 4)),
        "failure_rate": float(round(failure_rate, 4)),
        "average_reliability_score": float(round(average_reliability, 4)),
        "average_latency_seconds": float(round(avg_latency, 4)),
    }

    with open("llm_agent_reliability_summary.json", "w", encoding="utf-8") as file:
        json.dump(summary, file, indent=4)

    recommendations = create_reliability_recommendations(summary, result_df)
    with open("llm_failure_recovery_recommendations.txt", "w", encoding="utf-8") as file:
        file.write(recommendations)

    survey_df = simulate_human_trust_survey(summary["average_reliability_score"])
    survey_df.to_csv("simulated_human_trust_survey.csv", index=False)

    print("\nFiles saved:")
    print("llm_agent_reliability_evaluation_results.csv")
    print("llm_ticket_type_confusion_matrix.csv")
    print("llm_agent_reliability_summary.json")
    print("llm_failure_recovery_recommendations.txt")
    print("simulated_human_trust_survey.csv")

    return result_df


def simulate_human_trust_survey(average_reliability):
    np.random.seed(RANDOM_STATE)
    rows = []

    for participant_id in range(1, SIMULATED_PARTICIPANTS + 1):
        pre_trust = np.random.randint(3, 6)

        if average_reliability >= 0.80:
            change = np.random.choice([1, 2], p=[0.45, 0.55])
        elif average_reliability >= 0.50:
            change = np.random.choice([-1, 0, 1], p=[0.20, 0.45, 0.35])
        else:
            change = np.random.choice([-2, -1, 0], p=[0.35, 0.45, 0.20])

        post_trust = int(max(1, min(7, pre_trust + change)))

        rows.append({
            "participant_id": participant_id,
            "pre_interaction_trust_1_to_7": int(pre_trust),
            "post_interaction_trust_1_to_7": post_trust,
            "trust_change": post_trust - int(pre_trust),
            "perceived_reliability_condition": trust_label(average_reliability),
        })

    return pd.DataFrame(rows)


def create_reliability_recommendations(summary, result_df):
    accuracy = summary["accuracy"]
    reliability = summary["average_reliability_score"]
    failure_counts = result_df["failure_mode"].value_counts().to_dict()

    lines = [
        "LLM-Powered AI Agent Failure, Reliability, and Trust Analysis",
        "",
        f"Model used: {summary['llm_model']}",
        f"Accuracy: {accuracy}",
        f"Task success rate: {summary['task_success_rate']}",
        f"Failure rate: {summary['failure_rate']}",
        f"Average reliability score: {reliability}",
        "",
        "Failure mode counts:",
    ]

    for key, value in failure_counts.items():
        lines.append(f"- {key}: {value}")

    lines.extend([
        "",
        "Interpretation:",
        "The system is an LLM-powered business support agent because it uses an instruction-tuned language model to classify customer tickets and support workflow decisions.",
        "Low accuracy or low reliability indicates that autonomous execution should not be allowed for uncertain tickets.",
        "Human review is required for low-confidence and wrong-class predictions.",
        "",
        "Recommended improvements:",
        "1. Use a larger model such as google/flan-t5-base, google/flan-t5-large, Llama, Gemini, or GPT for stronger reasoning.",
        "2. Add few-shot examples in the prompt for every ticket type.",
        "3. Use retrieval-augmented generation so the agent can consult business policy documents.",
        "4. Keep human-in-the-loop approval for high-priority and low-confidence tickets.",
        "5. Collect real participant questionnaire data to replace the simulated trust survey.",
    ])

    return "\n".join(lines)


def predict_new_ticket(llm, data, subject, description, priority):
    labels = sorted(data["ticket_type"].unique())
    text = subject + " " + description
    predicted_type, confidence, raw_output = llm_classify_ticket(llm, text, priority, labels)
    mode = "success" if confidence >= CONFIDENCE_THRESHOLD else "low_confidence_success"
    recovery = error_recovery_action(mode, predicted_type, confidence)
    customer_response = generate_customer_response(llm, text, predicted_type, priority, recovery)

    print("\n========== NEW LLM TICKET PREDICTION ==========")
    print("Subject:", subject)
    print("Description:", description)
    print("Priority:", priority)
    print("Raw LLM output:", raw_output)
    print("Predicted ticket type:", predicted_type)
    print("Confidence:", confidence)
    print("Recommended business action:", business_action(predicted_type, priority, confidence))
    print("Error recovery action:", recovery)
    print("LLM-generated customer response:", customer_response)


data = load_and_prepare_data(DATASET_PATH)
llm = load_llm()
results = evaluate_llm_agent(data, llm)

predict_new_ticket(
    llm,
    data,
    subject="Payment failed but money deducted",
    description="Customer paid during checkout, amount was deducted, but order was not confirmed.",
    priority="High",
)

predict_new_ticket(
    llm,
    data,
    subject="Product stopped working",
    description="The device stopped working after two days and customer needs replacement.",
    priority="Medium",
)


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LLM loaded: google/flan-t5-small
Device: cpu

Evaluating LLM-powered AI agent...
Allowed ticket types: ['Billing inquiry', 'Cancellation request', 'Product inquiry', 'Refund request', 'Technical issue']
Evaluation records: 300
Processed 25/300 tickets
Processed 50/300 tickets
Processed 75/300 tickets
Processed 100/300 tickets
Processed 125/300 tickets
Processed 150/300 tickets
Processed 175/300 tickets
Processed 200/300 tickets
Processed 225/300 tickets
Processed 250/300 tickets
Processed 275/300 tickets
Processed 300/300 tickets

========== LLM AGENT DATASET SUMMARY ==========
Dataset used: Customer Support Ticket Dataset
Total records used: 8469
Training/reference records: 6351
Evaluated records: 300
Target column: ticket_type
Business task: LLM ticket classification, workflow routing, reliability evaluation

========== LLM AGENT PERFORMANCE ==========
Accuracy: 0.2033
Task success rate: 0.1867
Failure rate: 0.8133
Average reliability score: 0.226
Average latency per ticket: 0.1698 s